# Bekaert & Hoerova (2014) — HAR-RV: VIX vs Variance Swap vs Paper
## Daily-return proxy benchmarked against B&H (2014) original results

**Paper:** *"The VIX, the Variance Premium and Stock Market Volatility"*  
Geert Bekaert & Marie Hoerova, ECB Working Paper No. 1675, May 2014

---

### What We're Measuring

The paper uses 5-minute intraday RV with VIX²/12. We replicate using **daily squared returns** as the RV proxy with two implied variance measures and compare directly against the paper:

| | HAR-RV-VIX | HAR-RV-VS | **Paper (B&H 2014)** |
|---|---|---|---|
| Risk-neutral var | VIX²/12 | VS²/12 | VIX²/12 |
| RV proxy | $(r_t \times 100)^2$ daily sq | $(r_t \times 100)^2$ daily sq | **5-min intraday** |
| Sample | 2008–2026 | 2008–2026 | 1990–2010 |
| OOS split (75%) | 2021-10-27 | 2021-10-27 | 2005-07-15 |

**VIX** — CBOE model-free implied vol, computed from a discrete strip of SPX options. Carries a convexity/truncation upward bias relative to the true `E^Q[RV]`.  
**VS** — 1-month SPX variance swap fair vol strike: the exact breakeven where `VS² = E^Q[RV_annual]` under no-arbitrage. No approximation bias.

**Key constraint:** The paper's 5-min RV has ~78 observations/day vs our 1 → ~252× higher estimation variance. This is the primary source of all performance gaps vs the paper.

### Sections
1. [Setup](#1.-Setup)
2. [Data & VIX vs VS Descriptives](#2.-Data-&-VIX-vs-VS-Descriptives)
3. [RV Construction](#3.-RV-Construction)
4. [Unified Panel](#4.-Unified-Panel-Construction)
5. [Estimation Helpers](#5.-Estimation-Helpers)
6. [In-Sample Estimation — VIX, VS vs Paper](#6.-In-Sample-Estimation)
7. [Out-of-Sample Forecasting — VIX, VS vs Paper](#7.-Out-of-Sample-Forecasting)
8. [Variance Premium](#8.-Variance-Premium)
9. [Return Predictability](#9.-Return-Predictability)
10. [Rolling R²](#10.-Rolling-R²)
11. [Summary vs Paper](#11.-Summary)

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from statsmodels.stats.sandwich_covariance import cov_hac

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

DATA_DIR   = Path('..') / 'data'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

NW_LAGS        = 44       # Newey-West lags (paper convention)
PAPER_OOS_MZR2 = 0.555    # paper OOS MZ-R² (VIX, 5-min RV, 1990-2010)
PAPER_IS_RMSE  = 10.508   # paper IS RMSE
PAPER_OOS_RMSE = 46.077
PAPER_OOS_MAE  = 16.856
PAPER_OOS_MAPE = 0.347
PAPER_COEFS    = {'const': 3.730, 'X_lag': 0.108,
                  'RV22_lag': 0.199, 'RV5_lag': 0.330, 'RV1_lag': 0.107}

print('Setup complete.')

## 2. Data & VIX vs VS Descriptives

In [ ]:
# ── SP500 front-month futures returns ────────────────────────────────────────
meta = pd.read_parquet(DATA_DIR / 'EquityFuture_security_meta.parquet')
hist = pd.read_parquet(DATA_DIR / 'EquityFuture_historical.parquet')
es_tickers = meta[meta['curve_group'] == 'ES']['security'].tolist()
es = hist[hist['security'].isin(es_tickers)].copy()
es['date'] = pd.to_datetime(es['date'])
meta_es = meta[meta['curve_group'] == 'ES'][['security', 'expiry_yearmonth']].copy()
meta_es['expiry_date'] = pd.to_datetime(meta_es['expiry_yearmonth'], format='%Y-%m')
es = es.merge(meta_es[['security', 'expiry_date']], on='security').sort_values(['date', 'expiry_date'])
sp_ret = (es.groupby('date').first().reset_index()[['date', 'returns']]
          .dropna().sort_values('date').set_index('date')['returns'])

# ── VIX ──────────────────────────────────────────────────────────────────────
vix_df = pd.read_csv(DATA_DIR / 'VolatilityIndexData.csv', parse_dates=['DATE'])
vix = vix_df[vix_df['SECURITY'] == 'VIX Index'].sort_values('DATE').set_index('DATE')['INDEX_VALUE']
vix.index.name = 'date'

# ── Variance swap (1-month SPX) ───────────────────────────────────────────────
vs_raw = pd.read_csv(DATA_DIR / 'EquityIndexVarianceSwapData.csv', parse_dates=['DATE'])
vs = (vs_raw[(vs_raw['UNDERLYING'] == 'SPX') & (vs_raw['TENOR_MONTHS'] == 1.0)]
      .sort_values('DATE').set_index('DATE')['IMPLIED_VOLATILITY'])
vs.index.name = 'date'

print(f'SP500 returns : {len(sp_ret):,} obs  ({sp_ret.index.min().date()} – {sp_ret.index.max().date()})')
print(f'VIX           : {len(vix):,} obs  ({vix.index.min().date()} – {vix.index.max().date()})')
print(f'Variance swap : {len(vs):,} obs  ({vs.index.min().date()} – {vs.index.max().date()})')

In [ ]:
# ── VIX vs VS descriptive comparison ─────────────────────────────────────────
# Both expressed as monthly variance (annualised vol²/12)
vix2 = vix ** 2 / 12.0
vs2  = vs  ** 2 / 12.0
overlap = vix2.index.intersection(vs2.index)
diff = vix2.loc[overlap] - vs2.loc[overlap]

desc = pd.DataFrame({
    'VIX²/12 (monthly var)': vix2.loc[overlap].describe(),
    'VS²/12  (monthly var)': vs2.loc[overlap].describe(),
    'Diff  VIX−VS':          diff.describe(),
}).round(3)
print('Descriptive statistics (monthly %² units):')
display(desc)
print(f'Correlation VIX²/12 vs VS²/12: {vix2.loc[overlap].corr(vs2.loc[overlap]):.4f}')
print(f'VIX upward bias: {diff.mean():.3f} %²  '
      f'({diff.mean()/vs2.loc[overlap].mean()*100:.1f}% above VS on average)')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

ax = axes[0]
ax.plot(vs2.loc[overlap].index,  vs2.loc[overlap],  color='steelblue', lw=0.8,
        label='VS²/12 — variance swap (exact E^Q[RV])')
ax.plot(vix2.loc[overlap].index, vix2.loc[overlap], color='firebrick', lw=0.8,
        alpha=0.7, label='VIX²/12 — CBOE approximation')
ax.set_ylim(-5, 600)
ax.set_ylabel('Monthly variance (%²)')
ax.set_title('VIX²/12 vs VS²/12 — risk-neutral variance measures')
ax.legend(fontsize=9)

ax = axes[1]
ax.fill_between(diff.index, diff, 0, where=diff >= 0,
                color='salmon', alpha=0.6, label=f'VIX > VS  (mean = {diff.mean():.2f} %²)')
ax.fill_between(diff.index, diff, 0, where=diff < 0,
                color='steelblue', alpha=0.6, label='VS > VIX (rare)')
ax.axhline(diff.mean(), color='grey', ls='--', lw=1)
ax.set_ylabel('VIX²/12 − VS²/12  (%²)')
ax.set_title('VIX convexity/truncation premium over VS')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 3. RV Construction

Daily realized variance proxy: $RV_t = (r_t \times 100)^2$. All quantities in **monthly %²** units.

| Variable | Formula |
|---|---|
| `RV1`  | $(r_t\times100)^2 \times 22$ — daily RV, monthly-scaled |
| `RV5`  | $\overline{(r\times100)^2}_{5d} \times 22$ — 5-day avg, monthly-scaled |
| `RV22` | $\sum_{s=0}^{21}(r_{t-s}\times100)^2$ — 22-day rolling sum |

**Paper uses 5-min intraday:** ~78 squared returns per day → ~252× lower estimation variance than our daily-sq proxy.

In [ ]:
rv_daily = (sp_ret * 100) ** 2
rv = pd.DataFrame(index=sp_ret.index)
rv['RV1']  = rv_daily * 22
rv['RV5']  = rv_daily.rolling(5).mean() * 22
rv['RV22'] = rv_daily.rolling(22).sum()
print(rv.describe().round(2))

## 4. Unified Panel Construction

One panel, both predictors. Date range determined by VS availability (starts Nov 2008).

- **Target:** `RV22_fwd` = `RV22.shift(-22)` — forward 22-day realized variance
- **Predictors (lagged 1 day):** `VS2_lag`, `VIX2_lag`, `RV22_lag`, `RV5_lag`, `RV1_lag`

In [ ]:
panel = rv.join(vs2.rename('VS2'), how='inner')
panel['VIX2']     = vix2
panel['RV22_fwd'] = panel['RV22'].shift(-22)
panel['VS2_lag']  = panel['VS2'].shift(1)
panel['VIX2_lag'] = panel['VIX2'].shift(1)
panel['RV22_lag'] = panel['RV22'].shift(1)
panel['RV5_lag']  = panel['RV5'].shift(1)
panel['RV1_lag']  = panel['RV1'].shift(1)
panel = panel.dropna()

n          = len(panel)
SPLIT_DATE = panel.index[int(0.75 * n)]

print(f'Panel: {n:,} obs  ({panel.index.min().date()} – {panel.index.max().date()})')
print(f'75% OOS split at: {SPLIT_DATE.date()}  '
      f'(train={int(0.75*n):,}, test={n-int(0.75*n):,})')

## 5. Estimation Helpers

Model: $RV_{t+1}^{(22)} = c + \alpha \cdot X_t^2/12 + \beta^m RV_t^{(22)} + \beta^w RV_t^{(5)} + \beta^d RV_t^{(1)} + \varepsilon_t$

where $X$ is either **VS** or **VIX**. Estimated by OLS with 44-lag Newey-West HAC standard errors.

In [ ]:
def nw_se(res, nlags=NW_LAGS):
    return np.sqrt(np.diag(cov_hac(res, nlags=nlags)))

def estimate_har(pnl, x_col, label=''):
    y   = pnl['RV22_fwd']
    X   = add_constant(pnl[[x_col, 'RV22_lag', 'RV5_lag', 'RV1_lag']])
    res = OLS(y, X).fit()
    ses = nw_se(res)
    vrs = X.columns.tolist()
    return dict(label=label, n=int(res.nobs), x_col=x_col,
                params=dict(zip(vrs, res.params)),
                nw_se=dict(zip(vrs, ses)),
                t_stats=dict(zip(vrs, res.params / ses)),
                adj_r2=res.rsquared_adj,
                rmse=float(np.sqrt(res.mse_resid)),
                fitted=res.fittedvalues)

def oos_forecast(pnl, split, x_col, label=''):
    tr, te = pnl[pnl.index <= split], pnl[pnl.index > split]
    res    = OLS(tr['RV22_fwd'],
                 add_constant(tr[[x_col, 'RV22_lag', 'RV5_lag', 'RV1_lag']])).fit()
    X_te   = add_constant(te[[x_col, 'RV22_lag', 'RV5_lag', 'RV1_lag']], has_constant='add')
    y_te   = te['RV22_fwd']
    y_hat  = res.predict(X_te)
    mz     = OLS(y_te.values, add_constant(y_hat.values)).fit()
    err    = y_te - y_hat
    isr    = estimate_har(tr, x_col)
    return dict(label=label, n_train=len(tr), n_test=len(te),
                is_adj_r2=isr['adj_r2'], is_rmse=isr['rmse'],
                oos_mz_r2=float(mz.rsquared),
                oos_rmse=float(np.sqrt((err**2).mean())),
                oos_mae=float(err.abs().mean()),
                oos_mape=float((err.abs() / y_te).mean()),
                y_test=y_te, y_hat=y_hat, err=err)

print('Helpers defined.')

## 6. In-Sample Estimation

Both models estimated on the full panel. Coefficients compared side-by-side against **B&H (2014) Table 3**.

In [ ]:
res_vs  = estimate_har(panel, 'VS2_lag',  label='HAR-RV-VS')
res_vix = estimate_har(panel, 'VIX2_lag', label='HAR-RV-VIX')

# Side-by-side coefficient table vs paper
rows_coef = []
for v, v_label in [('const','const'), ('X_lag','α (X²/12)'),
                   ('RV22_lag','β^m RV22'), ('RV5_lag','β^w RV5'), ('RV1_lag','β^d RV1')]:
    vk_vs  = res_vs['x_col']  if v == 'X_lag' else v
    vk_vix = res_vix['x_col'] if v == 'X_lag' else v
    rows_coef.append({
        'Term':           v_label,
        'VS coef':        round(res_vs['params'][vk_vs],   4),
        'VS t-stat':      round(res_vs['t_stats'][vk_vs],  2),
        'VIX coef':       round(res_vix['params'][vk_vix], 4),
        'VIX t-stat':     round(res_vix['t_stats'][vk_vix],2),
        'Paper coef':     PAPER_COEFS.get(v, '—'),
    })

coef_df = pd.DataFrame(rows_coef).set_index('Term')

print(f'IS Adj-R²:  VS = {res_vs["adj_r2"]:.4f}   VIX = {res_vix["adj_r2"]:.4f}')
print(f'IS RMSE:    VS = {res_vs["rmse"]:.3f}   VIX = {res_vix["rmse"]:.3f}'
      f'   Paper = {PAPER_IS_RMSE}  (ratio VS/paper = {res_vs["rmse"]/PAPER_IS_RMSE:.1f}×)')
print()
coef_df

**Reading the coefficient table vs B&H (2014):**

- **α inflated vs paper (0.108):** VS α ≈ 0.60, VIX α ≈ 0.56 — both ~5–6× the paper's value. Classic errors-in-variables: the noisy daily-sq RV proxy forces more weight onto the lower-noise implied variance signal.
- **Lagged RV coefficients near zero:** Far below the paper's values (0.199 / 0.330 / 0.107). The daily proxy's high noise swamps the lagged-RV signal.
- **IS RMSE ~4× the paper's 10.5:** Directly attributable to daily-sq variance inflation — not model misspecification.
- **VS α slightly higher than VIX α:** Level effect only — VS²/12 has a lower mean, so the model compensates. The fitted contribution `α × mean(X²/12)` is nearly identical for both.

In [ ]:
# ── IS fit comparison ─────────────────────────────────────────────────────────
panel['CV_vs']  = res_vs['fitted']
panel['CV_vix'] = res_vix['fitted']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, cv_col, label, colour in [
    (axes[0], 'CV_vs',  f'HAR-RV-VS   adj-R²={res_vs["adj_r2"]:.4f}',  'steelblue'),
    (axes[1], 'CV_vix', f'HAR-RV-VIX  adj-R²={res_vix["adj_r2"]:.4f}', 'darkorange'),
]:
    ax.plot(panel.index, panel['RV22_fwd'], color='grey', lw=0.6, alpha=0.7, label='Actual RV')
    ax.plot(panel.index, panel[cv_col],     color=colour, lw=0.8, alpha=0.9, label=f'Fitted CV ({label})')
    ax.set_ylim(-10, 500); ax.set_ylabel('Monthly RV (%²)')
    ax.legend(fontsize=9)
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('IS fit: HAR-RV-VS (top) vs HAR-RV-VIX (bottom)', y=1.01)
plt.tight_layout(); plt.show()
print('Correlation of fitted values (VS vs VIX):',
      round(panel['CV_vs'].corr(panel['CV_vix']), 4))

## 7. Out-of-Sample Forecasting

Train: up to 2021-10-27 &nbsp;|&nbsp; Test: 2021-10-28 – 2026-02-27  
Primary metric: **Mincer-Zarnowitz R²** — regresses actual on forecast, tests forecast unbiasedness.  
All metrics compared against **B&H (2014) paper benchmarks**.

In [ ]:
oos_vs  = oos_forecast(panel, SPLIT_DATE, 'VS2_lag',  label='HAR-RV-VS')
oos_vix = oos_forecast(panel, SPLIT_DATE, 'VIX2_lag', label='HAR-RV-VIX')

oos_cmp = pd.DataFrame([
    {'Model': 'HAR-RV-VS',
     'IS Adj-R²':  round(oos_vs['is_adj_r2'],  4),
     'IS RMSE':    round(oos_vs['is_rmse'],    3),
     'OOS MZ-R²':  round(oos_vs['oos_mz_r2'],  4),
     'OOS RMSE':   round(oos_vs['oos_rmse'],    3),
     'OOS MAE':    round(oos_vs['oos_mae'],     3),
     'OOS MAPE':   round(oos_vs['oos_mape'],    4)},
    {'Model': 'HAR-RV-VIX',
     'IS Adj-R²':  round(oos_vix['is_adj_r2'], 4),
     'IS RMSE':    round(oos_vix['is_rmse'],   3),
     'OOS MZ-R²':  round(oos_vix['oos_mz_r2'], 4),
     'OOS RMSE':   round(oos_vix['oos_rmse'],   3),
     'OOS MAE':    round(oos_vix['oos_mae'],    3),
     'OOS MAPE':   round(oos_vix['oos_mape'],   4)},
    {'Model': 'Paper (B&H 2014)',
     'IS Adj-R²':  '~0.56',
     'IS RMSE':    PAPER_IS_RMSE,
     'OOS MZ-R²':  PAPER_OOS_MZR2,
     'OOS RMSE':   PAPER_OOS_RMSE,
     'OOS MAE':    PAPER_OOS_MAE,
     'OOS MAPE':   PAPER_OOS_MAPE},
]).set_index('Model')

print(f'OOS window: {oos_vs["y_test"].index.min().date()} – {oos_vs["y_test"].index.max().date()}')
print(f'n_test = {oos_vs["n_test"]:,}')
print(f'\nIS RMSE ratio — VS/paper: {oos_vs["is_rmse"]/PAPER_IS_RMSE:.2f}×   '
      f'VIX/paper: {oos_vix["is_rmse"]/PAPER_IS_RMSE:.2f}×')
print(f'OOS RMSE ratio — VS/paper: {oos_vs["oos_rmse"]/PAPER_OOS_RMSE:.2f}×   '
      f'VIX/paper: {oos_vix["oos_rmse"]/PAPER_OOS_RMSE:.2f}×')
print()
display(oos_cmp)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Time-series of OOS forecasts
ax = axes[0]
y_te = oos_vs['y_test']
ax.plot(y_te.index,               y_te.values,            color='grey',       lw=0.9, alpha=0.8, label='Actual RV')
ax.plot(oos_vs['y_hat'].index,    oos_vs['y_hat'].values,  color='steelblue',  lw=0.9, alpha=0.9,
        label=f'HAR-RV-VS   MZ-R²={oos_vs["oos_mz_r2"]:.3f}  RMSE={oos_vs["oos_rmse"]:.1f}')
ax.plot(oos_vix['y_hat'].index,   oos_vix['y_hat'].values, color='darkorange', lw=0.9, alpha=0.8,
        ls='--', label=f'HAR-RV-VIX  MZ-R²={oos_vix["oos_mz_r2"]:.3f}  RMSE={oos_vix["oos_rmse"]:.1f}')
ax.set_ylim(-5, 150); ax.set_ylabel('Monthly RV (%²)')
ax.set_title(f'OOS forecasts  [Paper benchmark: MZ-R²={PAPER_OOS_MZR2}, RMSE={PAPER_OOS_RMSE}]')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Absolute error comparison
ax = axes[1]
ax.plot(oos_vs['err'].index,  oos_vs['err'].abs(),  color='steelblue',  lw=0.7, alpha=0.7,
        label=f'|error| VS  MAE={oos_vs["oos_mae"]:.2f}')
ax.plot(oos_vix['err'].index, oos_vix['err'].abs(), color='darkorange', lw=0.7, alpha=0.7,
        ls='--', label=f'|error| VIX MAE={oos_vix["oos_mae"]:.2f}')
ax.axhline(PAPER_OOS_MAE, color='firebrick', ls=':', lw=1.2,
           label=f'Paper MAE benchmark = {PAPER_OOS_MAE}')
ax.set_ylabel('Absolute forecast error (%²)')
ax.set_title('Absolute errors vs paper MAE benchmark')
ax.legend(fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout(); plt.show()

## 8. Variance Premium

$$VP_{\text{VS},t} = \frac{VS_t^2}{12} - CV_{\text{VS},t} \qquad VP_{\text{VIX},t} = \frac{VIX_t^2}{12} - CV_{\text{VIX},t}$$

VP_VS is the theoretically cleaner measure: no approximation bias in the risk-neutral term.  
VP_VIX is inflated by the VIX upward bias (~1.95 %² on average), which flows directly into VP.

In [ ]:
panel['VP_vs']  = panel['VS2']  - panel['CV_vs']
panel['VP_vix'] = panel['VIX2'] - panel['CV_vix']

vp_cmp = pd.DataFrame({
    'VP_VS  (VS²/12 − CV_VS)':    panel['VP_vs'].describe(),
    'VP_VIX (VIX²/12 − CV_VIX)': panel['VP_vix'].describe(),
}).round(3)
display(vp_cmp)
print(f'Correlation VP_VS vs VP_VIX: {panel["VP_vs"].corr(panel["VP_vix"]):.4f}')
print(f'Mean difference (VP_VIX − VP_VS): {(panel["VP_vix"] - panel["VP_vs"]).mean():.3f} %²  '
      f'≈ VIX upward bias ({diff.mean():.2f} %²)')

In [ ]:
# Save VP series
panel[['VS2', 'VIX2', 'CV_vs', 'CV_vix', 'VP_vs', 'VP_vix', 'RV22_fwd']].to_csv(
    OUTPUT_DIR / 'vp_vs_vix_comparison.csv')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, col, colour, title in [
    (axes[0], 'VP_vs',  'steelblue',  f'VP_VS = VS²/12 − CV_VS  (mean = {panel["VP_vs"].mean():.2f} %²)'),
    (axes[1], 'VP_vix', 'darkorange', f'VP_VIX = VIX²/12 − CV_VIX  (mean = {panel["VP_vix"].mean():.2f} %²)'),
]:
    ax.fill_between(panel.index, panel[col], 0,
                    where=panel[col] >= 0, color=colour, alpha=0.5)
    ax.fill_between(panel.index, panel[col], 0,
                    where=panel[col] < 0,  color='salmon', alpha=0.5)
    ax.axhline(0, color='black', lw=0.5)
    ax.axhline(panel[col].mean(), color='grey', ls='--', lw=1)
    ax.set_ylabel('VP (%²)')
    ax.set_title(title)

axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 9. Return Predictability

Regress h-month forward annualised excess SP500 returns on VP:  
$r_{t\to t+h}^{\text{excess}} = a + b \cdot VP_t + \varepsilon_t$  
at horizons h ∈ {1, 3, 12} months. End-of-month observations, NW SEs with max(3, 2h) lags.

**Paper benchmarks (Table 4, Panel A — 1990–2010):** 1m t=2.56\*\*, 3m t=5.63\*\*\*, 12m t=0.77 (insig.)

In [ ]:
def pred_reg(pnl, sp_returns, horizon, vp_col):
    monthly = pnl.resample('ME').last()[[vp_col]].dropna()
    sp_m    = sp_returns.resample('ME').agg(lambda x: (1 + x).prod() - 1)
    fwd     = (np.exp(np.log(1 + sp_m).rolling(horizon).sum().shift(-horizon)) - 1) * (12/horizon) * 100
    monthly['ret'] = fwd
    monthly = monthly.dropna()
    if len(monthly) < 20: return None
    y   = monthly['ret']
    X   = add_constant(monthly[[vp_col]])
    res = OLS(y, X).fit()
    nw  = np.sqrt(np.diag(cov_hac(res, nlags=max(3, 2*horizon))))
    vi  = list(X.columns).index(vp_col)
    return {'h': horizon, 'n': len(monthly),
            'coef':   round(float(res.params[vp_col]), 4),
            'NW-SE':  round(float(nw[vi]), 4),
            't-stat': round(float(res.params[vp_col] / nw[vi]), 2),
            'adj_r2': round(float(res.rsquared_adj), 4)}

rows = []
for h in [1, 3, 12]:
    rv = pred_reg(panel, sp_ret, h, 'VP_vs')
    rk = pred_reg(panel, sp_ret, h, 'VP_vix')
    if rv and rk:
        rows.append({
            'Horizon': f'{h}m  (n={rv["n"]})',
            'VP_VS coef':   rv['coef'],   'VP_VS t':   rv['t-stat'], 'VP_VS adj-R²':  rv['adj_r2'],
            'VP_VIX coef':  rk['coef'],   'VP_VIX t':  rk['t-stat'], 'VP_VIX adj-R²': rk['adj_r2'],
        })

pred_df = pd.DataFrame(rows).set_index('Horizon')
print('Return predictability (VP_VS and VP_VIX separately):')
display(pred_df)
print('\nPaper benchmarks (B&H 2014, Table 4, 1990-2010 sample):')
print('  1m: t=2.56**  adj-R²=0.031')
print('  3m: t=5.63*** adj-R²=0.103')
print(' 12m: t=0.77    adj-R²=-0.002')

**Reading the results vs paper:**
- Both VP_VS and VP_VIX show significant return predictability at 1m and 3m horizons — directionally consistent with the paper.
- VP_VS has stronger t-stats at 3m and 12m, consistent with it being a cleaner measure (no convexity bias noise).
- Sample differences (2008–2026 vs 1990–2010) mean direct magnitude comparison is limited; the **sign and horizon pattern** (3m strongest) replicate the core paper finding.

## 10. Rolling R²

Rolling 252-day in-sample Adj-R² at 22-day steps. Shows regime variation and compares to the paper's reported level.

In [ ]:
def rolling_r2(pnl, x_col, window=252, step=22):
    rows_r = []
    for end in pnl.index[::step]:
        start = end - pd.DateOffset(days=int(window * 365 / 252))
        sub   = pnl[(pnl.index >= start) & (pnl.index <= end)].dropna(subset=[x_col, 'RV22_fwd'])
        if len(sub) < window // 2: continue
        r = estimate_har(sub, x_col)
        rows_r.append({'date': end, 'adj_r2': r['adj_r2']})
    return pd.DataFrame(rows_r).set_index('date')

print('Computing rolling R² ...')
roll_vs  = rolling_r2(panel, 'VS2_lag')
roll_vix = rolling_r2(panel, 'VIX2_lag')
print(f'Done.  VS mean={roll_vs["adj_r2"].mean():.4f}  VIX mean={roll_vix["adj_r2"].mean():.4f}')
print(f'Paper OOS MZ-R² benchmark: {PAPER_OOS_MZR2}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(roll_vs.index,  roll_vs['adj_r2'],  color='steelblue',  lw=1.2,
        label=f'HAR-RV-VS   (mean={roll_vs["adj_r2"].mean():.3f})')
ax.plot(roll_vix.index, roll_vix['adj_r2'], color='darkorange', lw=1.2, ls='--',
        label=f'HAR-RV-VIX  (mean={roll_vix["adj_r2"].mean():.3f})')
ax.axhline(PAPER_OOS_MZR2, color='firebrick', ls=':', lw=1.5,
           label=f'Paper OOS MZ-R² = {PAPER_OOS_MZR2}')
ax.set_ylim(-0.15, 0.90); ax.set_ylabel('Adj-R²')
ax.set_title('Rolling 252-day IS Adj-R²: VS vs VIX vs Paper benchmark')
ax.legend(fontsize=9)
for yr, lbl in [(2020, 'COVID'), (2022, '2022 hikes')]:
    t = pd.Timestamp(f'{yr}-01-01')
    if roll_vs.index.min() <= t:
        ax.axvline(t, color='crimson', ls=':', lw=0.8, alpha=0.7)
        ax.text(t, 0.85, f' {lbl}', color='crimson', fontsize=7, rotation=90, va='top')
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout(); plt.show()

## 11. Summary vs Paper

In [ ]:
summary = pd.DataFrame([
    {'Metric': 'Sample',                   'HAR-RV-VS': '2008–2026', 'HAR-RV-VIX': '2008–2026', 'Paper (B&H 2014)': '1990–2010'},
    {'Metric': 'RV proxy',                 'HAR-RV-VS': 'Daily sq', 'HAR-RV-VIX': 'Daily sq',  'Paper (B&H 2014)': '5-min intraday'},
    {'Metric': 'Implied var predictor',    'HAR-RV-VS': 'VS²/12',   'HAR-RV-VIX': 'VIX²/12',   'Paper (B&H 2014)': 'VIX²/12'},
    {'Metric': '─── In-Sample ───',        'HAR-RV-VS': '',          'HAR-RV-VIX': '',            'Paper (B&H 2014)': ''},
    {'Metric': 'IS Adj-R²',                'HAR-RV-VS': round(res_vs['adj_r2'],  4), 'HAR-RV-VIX': round(res_vix['adj_r2'], 4), 'Paper (B&H 2014)': '~0.56'},
    {'Metric': 'IS RMSE',                  'HAR-RV-VS': round(res_vs['rmse'],    3), 'HAR-RV-VIX': round(res_vix['rmse'],   3), 'Paper (B&H 2014)': PAPER_IS_RMSE},
    {'Metric': 'IS RMSE / paper (×)',      'HAR-RV-VS': round(res_vs['rmse']/PAPER_IS_RMSE, 2), 'HAR-RV-VIX': round(res_vix['rmse']/PAPER_IS_RMSE, 2), 'Paper (B&H 2014)': '1.0×'},
    {'Metric': 'α coef',                   'HAR-RV-VS': round(res_vs['params']['VS2_lag'],  4), 'HAR-RV-VIX': round(res_vix['params']['VIX2_lag'], 4), 'Paper (B&H 2014)': PAPER_COEFS['X_lag']},
    {'Metric': 'α / paper α (×)',          'HAR-RV-VS': round(res_vs['params']['VS2_lag']/PAPER_COEFS['X_lag'],   1), 'HAR-RV-VIX': round(res_vix['params']['VIX2_lag']/PAPER_COEFS['X_lag'], 1), 'Paper (B&H 2014)': '1.0×'},
    {'Metric': '─── Out-of-Sample ───',    'HAR-RV-VS': '',          'HAR-RV-VIX': '',            'Paper (B&H 2014)': ''},
    {'Metric': 'OOS MZ-R²',               'HAR-RV-VS': round(oos_vs['oos_mz_r2'], 4), 'HAR-RV-VIX': round(oos_vix['oos_mz_r2'], 4), 'Paper (B&H 2014)': PAPER_OOS_MZR2},
    {'Metric': 'OOS RMSE',                'HAR-RV-VS': round(oos_vs['oos_rmse'],  3), 'HAR-RV-VIX': round(oos_vix['oos_rmse'],  3), 'Paper (B&H 2014)': PAPER_OOS_RMSE},
    {'Metric': 'OOS MAE',                 'HAR-RV-VS': round(oos_vs['oos_mae'],   3), 'HAR-RV-VIX': round(oos_vix['oos_mae'],   3), 'Paper (B&H 2014)': PAPER_OOS_MAE},
    {'Metric': 'OOS MAPE',                'HAR-RV-VS': round(oos_vs['oos_mape'],  4), 'HAR-RV-VIX': round(oos_vix['oos_mape'],  4), 'Paper (B&H 2014)': PAPER_OOS_MAPE},
    {'Metric': 'OOS RMSE / paper (×)',     'HAR-RV-VS': round(oos_vs['oos_rmse']/PAPER_OOS_RMSE, 2), 'HAR-RV-VIX': round(oos_vix['oos_rmse']/PAPER_OOS_RMSE, 2), 'Paper (B&H 2014)': '1.0×'},
    {'Metric': '─── Variance Premium ───', 'HAR-RV-VS': '',          'HAR-RV-VIX': '',            'Paper (B&H 2014)': ''},
    {'Metric': 'VP mean (monthly %²)',     'HAR-RV-VS': round(panel['VP_vs'].mean(),  3), 'HAR-RV-VIX': round(panel['VP_vix'].mean(), 3), 'Paper (B&H 2014)': '~8.2 (1990-2010)'},
    {'Metric': '─── Return Pred (3m) ───', 'HAR-RV-VS': '',          'HAR-RV-VIX': '',            'Paper (B&H 2014)': ''},
    {'Metric': '3m VP t-stat',             'HAR-RV-VS': rows[1]['VP_VS t'],   'HAR-RV-VIX': rows[1]['VP_VIX t'],   'Paper (B&H 2014)': '5.63***'},
    {'Metric': '3m VP adj-R²',             'HAR-RV-VS': rows[1]['VP_VS adj-R²'], 'HAR-RV-VIX': rows[1]['VP_VIX adj-R²'], 'Paper (B&H 2014)': 0.103},
]).set_index('Metric')

print('=== Performance vs B&H (2014) Paper ===')
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# IS RMSE vs paper
ax = axes[0]
models = ['VS', 'VIX', 'Paper']
vals   = [res_vs['rmse'], res_vix['rmse'], PAPER_IS_RMSE]
colors = ['steelblue', 'darkorange', 'firebrick']
bars = ax.bar(models, vals, color=colors, width=0.5, alpha=0.8)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f'{v:.1f}', ha='center', fontsize=9)
ax.set_ylabel('IS RMSE (%²)')
ax.set_title('In-Sample RMSE\n(lower = better)')

# OOS MZ-R² vs paper
ax = axes[1]
vals = [oos_vs['oos_mz_r2'], oos_vix['oos_mz_r2'], PAPER_OOS_MZR2]
bars = ax.bar(models, vals, color=colors, width=0.5, alpha=0.8)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)
ax.set_ylim(0, 0.75)
ax.set_ylabel('OOS MZ-R²')
ax.set_title('OOS MZ-R²\n(higher = better)')

# OOS MAE vs paper
ax = axes[2]
vals = [oos_vs['oos_mae'], oos_vix['oos_mae'], PAPER_OOS_MAE]
bars = ax.bar(models, vals, color=colors, width=0.5, alpha=0.8)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.1, f'{v:.2f}', ha='center', fontsize=9)
ax.set_ylabel('OOS MAE (%²)')
ax.set_title('OOS MAE\n(lower = better)')

plt.suptitle('VS and VIX vs Paper (B&H 2014) — Effect of Daily-Return RV Proxy',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---

## Key Findings vs B&H (2014)

### 1. The daily-return RV proxy is the binding constraint
IS RMSE is **~4× the paper's 10.5** for both VS and VIX models. OOS RMSE is similarly inflated.  
The paper uses ~78 squared intraday returns per day vs our single daily squared return → ~252× higher estimation variance in the RV target. This single factor explains all quantitative gaps.

### 2. The α (implied variance weight) is inflated ~5–6× vs paper
Paper: α = 0.108. Ours: VS α ≈ 0.60, VIX α ≈ 0.56.  
This is the textbook **errors-in-variables** signature: noise in the lagged-RV predictors pushes OLS weight toward the lower-noise implied variance signal.

### 3. VIX and VS perform equivalently on forecasting metrics
IS Adj-R² and OOS metrics differ by < 0.001 between the two models. The 0.9953 correlation between VS²/12 and VIX²/12 means they carry essentially identical predictive content over the overlapping sample.

### 4. VP_VS is the cleaner variance premium measure
VP_VIX overstates the true risk premium by ~1.95 %² (the VIX upward bias flows directly into VP).  
VP_VS has stronger return predictability t-stats at 3m and 12m horizons — consistent with removing approximation-bias noise.

### 5. Return predictability pattern replicates
Both VP_VS and VP_VIX show significant 1m and 3m predictability — directionally consistent with Table 4 of B&H (2014). The 3m horizon is strongest, matching the paper's finding.

---

*Bekaert & Hoerova (2014) · Corsi (2009) · Bollerslev, Tauchen & Zhou (2009)*